In [0]:
from pyspark.sql.functions import col, regexp_replace, when, lower, trim

spark.sql("CREATE SCHEMA IF NOT EXISTS airbnb_barcelona_project.silver")

listings_bronze = spark.table("airbnb_barcelona_project.bronze.listings")
calendar_bronze = spark.table("airbnb_barcelona_project.bronze.calendar")
reviews_bronze = spark.table("airbnb_barcelona_project.bronze.reviews")

# Quick peek at raw price format before cleaning
display(listings_bronze.select("price").limit(10))
display(calendar_bronze.select("price").limit(10) if "price" in calendar_bronze.columns else None)

price
$409.00
$388.00
$397.13
$43.24
$226.13
null
$417.80
$275.50
$107.61
$209.00


In [0]:
listings_silver = (listings_bronze
    .withColumn("price_clean", regexp_replace(col("price"), "[$,]", "").cast("double"))
    .withColumn("host_response_rate_clean", regexp_replace(col("host_response_rate"), "%", "").cast("double"))
    .withColumn("host_acceptance_rate_clean", regexp_replace(col("host_acceptance_rate"), "%", "").cast("double"))
    .withColumn("host_is_superhost_bool", when(col("host_is_superhost") == "t", True)
                                          .when(col("host_is_superhost") == "f", False)
                                          .otherwise(None))
    .withColumn("instant_bookable_bool", when(col("instant_bookable") == "t", True)
                                        .when(col("instant_bookable") == "f", False)
                                        .otherwise(None))
    .withColumn("has_availability_bool", when(col("has_availability") == "t", True)
                                        .when(col("has_availability") == "f", False)
                                        .otherwise(None))
)

print(f"Rows: {listings_silver.count()}")
display(listings_silver.select("price", "price_clean", "host_response_rate", "host_response_rate_clean",
                                 "host_is_superhost", "host_is_superhost_bool").limit(10))

Rows: 15293


price,price_clean,host_response_rate,host_response_rate_clean,host_is_superhost,host_is_superhost_bool
$409.00,409.0,null,null,f,false
$388.00,388.0,null,null,t,true
$397.13,397.13,null,null,t,true
$43.24,43.24,null,null,f,false
$226.13,226.13,null,null,t,true
null,null,null,null,f,false
$417.80,417.8,null,null,f,false
$275.50,275.5,null,null,f,false
$107.61,107.61,null,null,f,false
$209.00,209.0,null,null,t,true


In [0]:
total = listings_silver.count()
null_prices = listings_silver.filter(col("price_clean").isNull()).count()
print(f"Total: {total}, Null prices: {null_prices} ({round(null_prices/total*100, 1)}%)")

Total: 15293, Null prices: 1938 (12.7%)


In [0]:
display(listings_silver.select("license").distinct().limit(30))

total = listings_silver.count()
null_license = listings_silver.filter(col("license").isNull()).count()
print(f"Total: {total}, Null license: {null_license} ({round(null_license/total*100, 1)}%)")

license
Spain - National registration numberESFCTU000008058000039706000000000000000HUTB-002062349Barcelona - Regional registration numberHUTB-002062
Spain - National registration numberESFCTU000008106000547360000000000000000000HUTB0050572Barcelona - Regional registration numberHUTB-005057
Spain - National registration numberESFCTU000008119000093652000000000000000HUTB-001506712Barcelona - Regional registration numberHUTB-150671
null
Barcelona - Regional registration numberHUTB-001282Spain - National registration numberESFCTU00000805400028019300000000000000000HUTB-0012821
Barcelona - Regional registration numberHUTB-001723Spain - National registration numberESFCTU00000805800030001100000000000000000HUTB-0017235
Barcelona - Regional registration numberHUTB-007966
Barcelona - Regional registration numberHUTB-002424Spain - National registration numberESFCTU00000807300018946600000000000000HUTB-002424-083
Spain - National registration numberESFCNT00000805600004336500000000000000000000000000009Barcelona - Regional registration numberExempt - seasonal rental
Barcelona - Regional registration numberHUTB-000173Spain - National registration numberESFCTU00000807200078128100000000000000000HUTB-0001713


Total: 15293, Null license: 3043 (19.9%)


In [0]:
# Are null-price listings concentrated in a particular status (e.g. inactive/unavailable listings)?
display(listings_silver.filter(col("price_clean").isNull())
        .groupBy("has_availability")
        .count())

display(listings_silver.filter(col("price_clean").isNull())
        .select("number_of_reviews", "availability_365")
        .summary("count", "mean", "min", "max"))

has_availability,count
t,1858
null,68
f,12


summary,number_of_reviews,availability_365
count,1938,1938
mean,28.49484004127967,39.642414860681114
min,0,0
max,1774,365


In [0]:
display(listings_silver.groupBy("license").count().orderBy(col("count").desc()).limit(20))

license,count
null,3043
Barcelona - Regional registration numberExempt - seasonal rental,194
Spain - National registration numberExempt - tourist apartmentBarcelona - Regional registration numberExempt - seasonal rental,118
Spain - National registration numberESFCNT00000805900063465600000000000000000000000000001Barcelona - Regional registration numberExempt - seasonal rental,41
Barcelona - Regional registration numberExempt - seasonal rentalSpain - National registration numberExempt - tourist apartment,25
Barcelona - Regional registration numberHUTB012407,24
Barcelona - Regional registration numberAJ000593Spain - National registration numberExempt - hostel,22
Spain - National registration numberESFCTU00000807200005651800000000000000000000000071101Barcelona - Regional registration numberHUTB-007110,20
Barcelona - Regional registration numberHUTB-012457Spain - National registration numberESFCTU00000812000014847100000000000000000HUTB-0124575,20
Barcelona - Regional registration numberHB-004525,17


In [0]:
from pyspark.sql.functions import col, when

listings_silver = listings_silver.withColumn(
    "license_status",
    when(col("license").isNull(), "No License Shown")
    .when(col("license").contains("Exempt"), "Exempt")
    .when(col("license").rlike("HUTB|HB-|ESFCNT|AJ0"), "Licensed")
    .otherwise("Other/Unclear")
)

display(listings_silver.groupBy("license_status").count().orderBy(col("count").desc()))

license_status,count
Licensed,7844
Exempt,3924
No License Shown,3043
Other/Unclear,482


In [0]:
display(listings_silver.filter(col("license_status") == "Other/Unclear").select("license").limit(10))

license
Spain - National registration numberESHFNT00000807200002789100100000000000000000000000001
Spain - National registration numberESHFNT00000805600009410700200000000000000000000000006
Spain - National registration numberESHFNT00000806600032689400100000000000000000000000009
Spain - National registration numberESHFNT00000806300015064100200000000000000000000000000
Barcelona - Regional registration numberhutb-003607Spain - National registration numberESFCTU000008054000290185000000000000000000HUBT0036077
Spain - National registration numberESHFNT00000806800020999300100000000000000000000000001
Barcelona - Regional registration numberhutb-009206Spain - National registration numberESFCTU0000080660002875770000000000000000000009206-390
Spain - National registration numberESHFNT00000813500046296000100000000000000000000000001
Barcelona - Regional registration numberHutb-006600
Barcelona - Regional registration numberATB-000089Spain - National registration numberESFCTU000008073000875017000000000000000000ATB-0000897


In [0]:
display(listings_silver.select("neighbourhood_cleansed").distinct().count())
display(listings_silver.groupBy("neighbourhood_cleansed").count().orderBy(col("count").desc()).limit(15))

69

neighbourhood_cleansed,count
la Dreta de l'Eixample,2145
el Raval,1031
"Sant Pere, Santa Caterina i la Ribera",925
la Vila de Gràcia,924
la Sagrada Família,910
el Barri Gòtic,881
l'Antiga Esquerra de l'Eixample,850
Sant Antoni,846
la Nova Esquerra de l'Eixample,660
el Poble Sec,650


In [0]:
from pyspark.sql.functions import col, when

listings_silver = listings_silver.withColumn(
    "license_status",
    when(col("license").isNull(), "No License Shown")
    .when(col("license").contains("Exempt"), "Exempt")
    .when(col("license").rlike("(?i)HUTB|HB-|ESFCNT|AJ0|ESHFNT|ATB-"), "Licensed")
    .otherwise("Other/Unclear")
)

display(listings_silver.groupBy("license_status").count().orderBy(col("count").desc()))

license_status,count
Licensed,8288
Exempt,3924
No License Shown,3043
Other/Unclear,38


In [0]:
print(listings_silver.select("neighbourhood_cleansed").distinct().count())

69


In [0]:
listings_silver_final = listings_silver.select(
    "id", "host_id", "host_name", "host_since", "host_is_superhost_bool",
    "host_response_rate_clean", "host_acceptance_rate_clean",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bedrooms", "beds",
    "price_clean", "minimum_nights", "maximum_nights",
    "availability_30", "availability_365",
    "number_of_reviews", "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value", "reviews_per_month",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
    "license_status", "instant_bookable_bool"
).withColumnRenamed("price_clean", "price") \
 .withColumnRenamed("host_response_rate_clean", "host_response_rate") \
 .withColumnRenamed("host_acceptance_rate_clean", "host_acceptance_rate") \
 .withColumnRenamed("host_is_superhost_bool", "host_is_superhost") \
 .withColumnRenamed("instant_bookable_bool", "instant_bookable")

listings_silver_final.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.silver.listings")

print(f"Silver listings: {listings_silver_final.count()} rows")

Silver listings: 15293 rows


In [0]:
from pyspark.sql.functions import col, when, regexp_replace

# Calendar: convert "t"/"f" to boolean, clean price if present
calendar_silver = (calendar_bronze
    .withColumn("available_bool", when(col("available") == "t", True)
                                 .when(col("available") == "f", False)
                                 .otherwise(None))
    .select("listing_id", "date", "available_bool", "minimum_nights", "maximum_nights")
    .withColumnRenamed("available_bool", "available")
)

calendar_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.silver.calendar")

print(f"Silver calendar: {calendar_silver.count()} rows")

Silver calendar: 5623190 rows


In [0]:
# Reviews: strip HTML tags from comments, drop rows with null core fields
reviews_silver = (reviews_bronze
    .withColumn("comments_clean", regexp_replace(col("comments"), "<br\\s*/?>", " "))
    .filter(col("listing_id").isNotNull() & col("id").isNotNull())
    .select("id", "listing_id", "date", "reviewer_id", "reviewer_name", "comments_clean")
    .withColumnRenamed("comments_clean", "comments")
)

reviews_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("airbnb_barcelona_project.silver.reviews")

print(f"Silver reviews: {reviews_silver.count()} rows")

Silver reviews: 1033523 rows
